# Fixed-duration reconstruction and the EVREAL cross-check
Results viewer for `benchmarks/recon_eval_tms.py` (the study) and `benchmarks/recon_evreal_tms.py` (EVREAL, unchanged). Run the scripts in a terminal first; this notebook only reads their reports.

```
python benchmarks/recon_eval_tms.py eds --bin-ms 2 5 10 20 --devices cuda:0 cuda:0
python benchmarks/recon_evreal_tms.py eds --bin-ms 2 5 10 20 --gpu 0
```

**Study (`recon_eval_tms.py`)**
- Back-to-back windows of 5 × `bin_ms`, starting at the first frame for every source, so window *i* covers the same time for real events and every method.
- `ms` settings use fixed bin edges (bin *b* = [b·D, (b+1)·D) ms of the window); `events` settings stretch the bins between the window's first and last event, as EVREAL does.
- **Main metric: agreement** of each method's reconstruction with the reconstruction from real events, at every window. Ground-truth scores only exist for windows ending within `gt_tol_ms` of a frame.

**Cross-check (`recon_evreal_tms.py`)**: EVREAL's own `eval.py` on the same events: ground-truth scores only, windows starting at each stream's own first event, plus EVREAL's standard between-frames run to compare with `recon_eval.py`.

In [ ]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage

REPO = Path.cwd().resolve()
while not (REPO / "benchmarks" / "recon_eval_tms.py").is_file():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "benchmarks"))
import recon_eval

# ---- settings -------------------------------------------------------------
DATASET = "eds"            # folder name under data_test_evt/
SPLIT = "test"
TMS_RUN = "latest"         # recon_eval_tms.py report folder, or "latest"
EVREAL_RUN = "latest"      # recon_evreal_tms.py report folder, "latest", or None to skip
RECON_RUN = "latest"       # recon_eval.py report folder (between frames), "latest", or None to skip
METRIC = "lpips"           # agreement metric for the main table and chart: "lpips", "ssim" or "mse"
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 250)


def resolve(run, base, marker):
    """A report folder, or the newest one under base that contains marker when run == 'latest'."""
    if run is None:
        return None
    if run != "latest":
        path = Path(run) if Path(run).is_absolute() else REPO / run
        if not (path / marker).is_file():
            raise FileNotFoundError(f"{path} has no {marker}")
        return path
    runs = sorted(p for p in base.glob("*") if (p / marker).is_file())
    if not runs:
        print(f"No finished run under {base}")
        return None
    return runs[-1]


tms_dir = resolve(TMS_RUN, REPO / "reports" / "recon_tms" / DATASET / SPLIT, "summary_all.csv")
evreal_dir = resolve(EVREAL_RUN, REPO / "reports" / "recon_evreal" / DATASET / SPLIT, "evreal_summary.csv")
recon_dir = resolve(RECON_RUN, REPO / "reports" / "recon" / DATASET / SPLIT, "summary.csv")
REAL = recon_eval.REAL
EVTWEEN = set(recon_eval.EVTWEEN)
HIGHER = recon_eval.HIGHER_IS_BETTER
order = {m: i for i, m in enumerate((REAL, *recon_eval.METHODS))}
NO_STUDY = ("No finished recon_eval_tms.py run: set TMS_RUN to a run folder, wait for the run to finish, or "
            "summarise a run in progress with: python benchmarks/recon_eval_tms.py " + DATASET + " --summarize-only <run folder>")
print("study:      ", tms_dir)
print("EVREAL:     ", evreal_dir)
print("recon_eval: ", recon_dir)

In [ ]:
if tms_dir is None:
    print(NO_STUDY)
else:
    config = json.loads((tms_dir / "config.json").read_text())
    summary = pd.read_csv(tms_dir / "summary_all.csv")
    configs = sorted(summary["config"].unique(), key=lambda c: (c.split("_")[-1] != "ms", float(c[3:].split("ms")[0])))
    print("methods:", config["methods"])
    print("settings:", configs, "| warm-up", config["warmup_s"], "s | GT tolerance", config["gt_tol_ms"], "ms")


    def cell(row, col):
        if pd.isna(row.get(f"{col}_mean")):
            return ""
        std = row.get(f"{col}_std")
        std = f" ± {std:.3f}" if pd.notna(std) else ""
        return f"{row[f'{col}_mean']:.3f}{std} ({int(row[f'{col}_rank'])})"


    def by_setting(col, include_real=False):
        """Rows = network, method; columns = settings; 'mean ± std over sequences (rank)'."""
        s = summary if include_real else summary[summary["method"] != REAL]
        t = s.assign(value=s.apply(lambda r: cell(r, col), axis=1)).pivot_table(
            index=["network", "method"], columns="config", values="value", aggfunc="first")[configs]
        t = t.reset_index()
        t["_o"] = t["method"].map(order)
        return t.sort_values(["network", "_o"]).drop(columns="_o").set_index(["network", "method"])

## Main result: agreement with the reconstruction from real events
Mean ± std over sequences (each sequence weighted equally), rank within each setting and network in brackets. LPIPS and MSE: lower is better; SSIM: higher is better.

In [ ]:
if tms_dir is None:
    print(NO_STUDY)
else:
    col = f"agreement_{METRIC}"
    print(f"{col} ({'higher' if HIGHER[METRIC] else 'lower'} is better)")
    display(by_setting(col))

## How each method holds up as bins get shorter
One panel per network and bin mode. Evtween variants are in colour; baselines are gray, named at the right end of each line.

In [ ]:
if tms_dir is None:
    print(NO_STUDY)
else:
    # reference palette categorical slots (light mode), in fixed order; gray for context lines
    SLOTS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
    GRAY, INK, MUTED = "#a3a29c", "#0b0b0b", "#52514e"
    col = f"agreement_{METRIC}_mean"
    data = summary[summary["method"] != REAL]
    nets = [n for n in recon_eval.NETWORKS if n in set(data["network"])]
    modes = [m for m in ("ms", "events") if m in set(data["bin_time"])]
    ours = [m for m in recon_eval.METHODS if m in EVTWEEN and m in set(data["method"])]
    color = {m: SLOTS[i] for i, m in enumerate(ours[:len(SLOTS)])}   # fixed per method, independent of rank

    fig, axes = plt.subplots(len(nets), len(modes), figsize=(5.2 * len(modes), 3.6 * len(nets)),
                             sharex=True, sharey=True, squeeze=False)
    for i, net in enumerate(nets):
        for j, mode in enumerate(modes):
            ax = axes[i][j]
            d = data[(data["network"] == net) & (data["bin_time"] == mode)]
            ends = []
            for method, g in sorted(d.groupby("method"), key=lambda kv: (kv[0] in color, order.get(kv[0], 99))):
                g = g.sort_values("bin_ms")
                ours_line = method in color
                ax.plot(g["bin_ms"], g[col], color=color.get(method, GRAY), lw=2 if ours_line else 1.2,
                        marker="o", ms=5 if ours_line else 3.5, zorder=3 if ours_line else 2,
                        label=method if ours_line else None)
                ends.append([g[col].iloc[-1], g["bin_ms"].iloc[-1], method, INK if ours_line else MUTED])
            # end-of-line labels, pushed apart where lines end close together
            lo, hi = ax.get_ylim()
            gap = 0.045 * (hi - lo)
            ends.sort(key=lambda e: e[0])
            placed = []
            for y, x, method, ink in ends:
                y_lab = max(y, placed[-1] + gap) if placed else y
                placed.append(y_lab)
                ax.annotate(method, (x, y), xytext=(x * 1.08, y_lab), textcoords="data", va="center", fontsize=7.5,
                            color=ink, annotation_clip=False)
            ax.set_ylim(lo, max(hi, placed[-1] + gap) if placed else hi)
            ax.set_xscale("log")
            ticks = sorted(d["bin_ms"].unique())
            ax.set_xticks(ticks); ax.set_xticklabels([f"{t:g}" for t in ticks]); ax.minorticks_off()
            ax.set_title(f"{net} · {'fixed-ms bins' if mode == 'ms' else 'EVREAL-style bins'}", fontsize=10, color=INK)
            ax.grid(axis="y", color="#e6e5e0", lw=0.8); ax.set_axisbelow(True)
            for side in ("top", "right"):
                ax.spines[side].set_visible(False)
            for side in ("left", "bottom"):
                ax.spines[side].set_color("#c9c8c2")
            ax.tick_params(colors=MUTED, labelsize=8)
            if j == 0:
                ax.set_ylabel(f"agreement {METRIC.upper()} ({'higher' if HIGHER[METRIC] else 'lower'} = closer to real)",
                              fontsize=8.5, color=MUTED)
            if i == len(nets) - 1:
                ax.set_xlabel("bin length (ms); window = 5 bins", fontsize=8.5, color=MUTED)
            ax.margins(x=0.25)
    handles, labels = axes[0][0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="upper center", ncol=min(4, len(handles)), frameon=False, fontsize=8.5)
    fig.tight_layout(rect=(0, 0, 1, 0.93 if handles else 1))
    out = tms_dir / f"agreement_{METRIC}_vs_bin_length.png"
    fig.savefig(out, dpi=150)
    plt.show()
    print("saved", out)
    print("Table view:")
    display(pd.read_csv(tms_dir / "comparison.csv").set_index(["network", "bin_time", "method"]))

## Secondary: against ground-truth frames
Only windows ending within `gt_tol_ms` of a frame are scored; `gt_windows` says how many per setting. The `real` row is the reference level. Simulators were given these frames, so a simulator can beat `real` here without being more realistic.

In [ ]:
if tms_dir is None:
    print(NO_STUDY)
else:
    display(summary.pivot_table(index=["network", "method"], columns="config", values="gt_windows", aggfunc="first")[configs]
            .rename(columns=lambda c: f"{c} (GT windows)").head(3))
    for m in ("lpips", "ssim", "mse"):
        print(f"gt_{m}")
        display(by_setting(f"gt_{m}", include_real=True))

## Sanity check of the inputs
Empty windows per source and setting. Many empty windows mean short bins are starved of events for that source, which hurts every reconstruction from it.

In [ ]:
if tms_dir is None:
    print(NO_STUDY)
else:
    sanity = pd.concat([pd.read_csv(tms_dir / c / "sanity.csv") for c in configs if (tms_dir / c / "sanity.csv").is_file()])
    ok = sanity[sanity["status"] == "ok"]
    table = ok.groupby(["config", "method"]).agg(events=("events", "sum"), windows=("windows", "sum"),
                                                 empty=("empty_windows", "sum")).reset_index()
    table["empty %"] = (100 * table["empty"] / table["windows"]).round(1)
    display(table.pivot_table(index="method", columns="config", values="empty %")[configs]
            .sort_index(key=lambda s: s.map(order)))
    display(sanity[sanity["status"] != "ok"])

## Best Evtween variant vs best baseline
Paired Wilcoxon signed-rank test over sequences, per setting, network and metric. With fewer than 6 sequences the two-sided p-value cannot go below 0.05.

In [ ]:
if tms_dir is None:
    print(NO_STUDY)
else:
    wx = pd.read_csv(tms_dir / "wilcoxon_all.csv")
    display(wx[wx["metric"].str.startswith("agreement")])

## EVREAL cross-check
EVREAL's own scores against ground-truth frames (frame-weighted means, as EVREAL reports them). Window lengths match the study's settings; `between_frames` is EVREAL's standard protocol.

In [ ]:
if evreal_dir is not None:
    ev = pd.read_csv(evreal_dir / "evreal_summary.csv")
    metrics = [c.removesuffix("_frame_weighted") for c in ev.columns if c.endswith("_frame_weighted")]
    for m in metrics:
        print(f"EVREAL {m} ({'higher' if HIGHER.get(m) else 'lower'} is better)")
        t = ev.pivot_table(index=["network", "method"], columns="window_ms", values=f"{m}_frame_weighted",
                           aggfunc="first").round(4)
        cols = sorted(t.columns, key=lambda c: (str(c) != "between_frames", float(c) if str(c) != "between_frames" else 0))
        t = t[cols].rename(columns=lambda c: c if str(c) == "between_frames" else f"{float(c):g} ms window")
        t = t.reset_index(); t["_o"] = t["method"].map(order)
        display(t.sort_values(["network", "_o"]).drop(columns="_o").set_index(["network", "method"]))
    off = pd.read_csv(evreal_dir / "first_event_offsets.csv") if (evreal_dir / "first_event_offsets.csv").is_file() else None
    if off is not None and not off.empty:
        print("First event of each stream after the first frame (ms): EVREAL starts its windows there")
        display(off.pivot_table(index="source", columns="sequence", values="first_event_offset_ms").round(3))
else:
    print("No EVREAL run selected")

### EVREAL `std` vs `recon_eval.py` (between frames)
Both use EVREAL's models and voxel rules between consecutive frames. Expect close but not identical numbers: `recon_eval.py` skips the first frames as warm-up, and LPIPS comes from different packages (`lpips` vs `pyiqa`).

In [ ]:
if evreal_dir is not None and recon_dir is not None:
    ev = pd.read_csv(evreal_dir / "evreal_summary.csv")
    std = ev[ev["window_ms"] == "between_frames"]
    pf = pd.read_csv(recon_dir / "per_frame.csv")
    ours = pf.groupby(["network", "method"])[["gt_mse", "gt_ssim", "gt_lpips"]].mean()  # frame-weighted, like EVREAL
    if std.empty:
        print("The EVREAL run has no between_frames results (it was run with --no-std)")
    else:
        theirs = std.set_index(["network", "method"])[[f"{m}_frame_weighted" for m in ("mse", "ssim", "lpips")
                                                        if f"{m}_frame_weighted" in std]]
        theirs.columns = [c.replace("_frame_weighted", "") + " EVREAL" for c in theirs.columns]
        ours.columns = [c.replace("gt_", "") + " recon_eval" for c in ours.columns]
        both = ours.join(theirs, how="inner")
        both = both[sorted(both.columns)].round(4).reset_index()
        both["_o"] = both["method"].map(order)
        display(both.sort_values(["network", "_o"]).drop(columns="_o").set_index(["network", "method"]))
else:
    print("Needs both an EVREAL run and a recon_eval.py run")

## Contact sheets of the selected windows
`selection.csv` lists the 40 windows per sequence and setting and why each was chosen: best or worst agreement, most real events (activity), or evenly spread (temporal). Each sheet shows the nearest ground-truth frame and its time offset.

In [ ]:
if tms_dir is None:
    print(NO_STUDY)
else:
    selection = pd.read_csv(tms_dir / "selection.csv") if (tms_dir / "selection.csv").is_file() else pd.DataFrame()
    if selection.empty:
        print("No selected windows (run with --save hybrid)")
    else:
        display(selection.groupby(["config", "category"]).size().unstack(fill_value=0))
        SHEET_CONFIG = configs[0]
        SHEET_SEQUENCE = selection["sequence"].iloc[0]
        SHEET_CATEGORY = "worst_agreement"   # "best_agreement" | "worst_agreement" | "high_activity" | "temporal"
        SHEET_LIMIT = 3
        for path in sorted((tms_dir / SHEET_CONFIG / "contact_sheets" / SHEET_SEQUENCE).glob(f"*_{SHEET_CATEGORY}.png"))[:SHEET_LIMIT]:
            print(path.name)
            display(IPImage(filename=str(path)))